ERROR: Could not find a version that satisfies the requirement torch==2.3.0 (from versions: none)
ERROR: No matching distribution found for torch==2.3.0

[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: C:\Users\user\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import os, gc, torch
os.environ["TOKENIZERS_PARALLELISM"] = "false"
torch.backends.cuda.matmul.allow_tf32 = True    # no effect on TPU but harmless
print("ℹ️ Torch-XLA ready")


ModuleNotFoundError: No module named 'torch'

In [3]:
import pandas as pd

conv_train_df = pd.read_csv('/kaggle/input/mental-health/conversations_training.csv')
mh_conv_df    = pd.read_csv('/kaggle/input/mental-health/mental_health_conversations.csv')
dialogues_df  = pd.read_csv('/kaggle/input/mental-health/dialogues_training.csv')
reddit_df     = pd.read_csv('/kaggle/input/mental-health/reddit_mental_health_combined.csv')
sentiment_df  = pd.read_csv('/kaggle/input/mental-health/sentiment_analysis.csv')


In [4]:
# ---------- 1. core Q & A tables -----------------------------------
all_convs = []

if {'input','output'}.issubset(conv_train_df.columns):
    tmp = conv_train_df[['input','output']].rename(
            columns={'input':'question','output':'response'})
    tmp['source'] = 'conversations_training'
    all_convs.append(tmp)

if {'question','answer'}.issubset(mh_conv_df.columns):
    tmp = mh_conv_df.rename(columns={'answer':'response'})
    tmp['source'] = 'mental_health_conversations'
    all_convs.append(tmp)

# ---------- 2. dialogue split on "__eou__" -------------------------
turns = []
for row in dialogues_df.itertuples():
    if pd.notna(row.text) and '__eou__' in row.text:
        seg = row.text.split('__eou__')
        for i in range(0, len(seg)-1, 2):
            if i+1 < len(seg):
                turns.append({'question':seg[i].strip(),
                              'response':seg[i+1].strip(),
                              'source':'dialogues'})
dialogue_df = pd.DataFrame(turns)
all_convs.append(dialogue_df)

# ---------- 3. sentiment synthetic responses -----------------------
sent_rows = []
for row in sentiment_df.sample(min(50_000, len(sentiment_df)),
                               random_state=42).itertuples():
    if pd.notna(row.text) and len(row.text) > 20:
        sent_rows.append({
            'question' : str(row.text)[:300],
            'response' : ("I hear you and understand you're expressing "
                          "important feelings. Would you like to explore "
                          "these emotions further?"),
            'source'   : 'sentiment_support'
        })
all_convs.append(pd.DataFrame(sent_rows))

# ---------- 4. reddit synthetic responses --------------------------
templ = [
 "Thank you for sharing this...",
 "I understand this must be challenging...",
 "It sounds like you're going through a difficult time...",
 "I appreciate you sharing your experience...",
 "Your feelings matter..."
]
red_rows = []
for i, row in enumerate(reddit_df.itertuples()):
    if pd.notna(row.text) and len(row.text) > 20:
        q = f"{row.title or ''} {row.text}".strip()[:500]
        red_rows.append({'question':q,
                         'response':templ[i % len(templ)],
                         'source':'reddit_posts'})
all_convs.append(pd.DataFrame(red_rows))

# ---------- 5. concatenate & clean ---------------------------------
df = pd.concat(all_convs, ignore_index=True)
df = df.dropna(subset=['question','response'])
df = df[df.question.str.len() > 5]
df = df[df.response.str.len() > 5]

print("Final rows:", len(df))


Final rows: 177379


In [5]:
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(df, test_size=0.10, random_state=42)
print(len(train_df), "train rows  |", len(val_df), "val rows")


159641 train rows  | 17738 val rows


In [6]:
from datasets import Dataset

def build_hf_ds(df, tok, max_len=512, num_proc=4):
    print("⏳ packing …")
    ds = Dataset.from_pandas(df[['question','response']])

    def to_chat(ex):
        ex["text"] = tok.apply_chat_template(
             [{"role":"user","content":ex['question']},
              {"role":"assistant","content":ex['response']}],
             tokenize=False, add_generation_prompt=False)
        return ex

    ds = ds.map(to_chat, num_proc=num_proc, remove_columns=ds.column_names)

    ds = ds.map(
        lambda e: tok(e["text"], add_special_tokens=True,
                      truncation=False, return_attention_mask=False),
        batched=True, num_proc=num_proc, remove_columns=['text'])

    def pack(ex):
        flat = sum(ex['input_ids'], [])
        n    = len(flat) // max_len * max_len
        blocks = [flat[i:i+max_len] for i in range(0, n, max_len)]
        return {'input_ids':blocks, 'labels':blocks}

    return ds.map(pack, batched=True, num_proc=num_proc,
                  remove_columns=ds.column_names).shuffle(seed=42)


/usr/local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
hf_token     = user_secrets.get_secret("HF_TOKEN")   # same secret you used before
login(token=hf_token, add_to_git_credential=True)

print("✓ Hugging Face token loaded")


Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when pushing to the Hugging Face Hub.
Run the following command in your terminal in case you want to set the 'store' credential helper as default.

git config --global credential.helper store

Read https://git-scm.com/book/en/v2/Git-Tools-Credential-Storage for more details.


Token has not been saved to git credential helper.


✓ Hugging Face token loaded


In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

BASE = "meta-llama/Llama-3.2-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(BASE, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
            BASE, torch_dtype=torch.bfloat16)

lora_cfg = LoraConfig(
    r=64, lora_alpha=16, lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    bias="none", task_type="CAUSAL_LM")
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()



Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]


Loading checkpoint shards:  50%|█████     | 1/2 [00:00<00:00,  2.86it/s]


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  3.53it/s]


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  3.40it/s]

trainable params: 36,700,160 || all params: 3,249,449,984 || trainable%: 1.1294


In [9]:
train_dataset = build_hf_ds(train_df, tokenizer)
eval_dataset  = build_hf_ds(val_df, tokenizer).select(range(2_000))
print("Blocks:", len(train_dataset), "train  |", len(eval_dataset), "eval")


⏳ packing …



Map (num_proc=4):   0%|          | 0/159641 [00:00<?, ? examples/s]


Map (num_proc=4):   0%|          | 72/159641 [00:01<1:10:17, 37.83 examples/s]


Map (num_proc=4):   1%|          | 1385/159641 [00:02<02:51, 920.53 examples/s]


Map (num_proc=4):   1%|▏         | 2299/159641 [00:02<01:35, 1643.39 examples/s]


Map (num_proc=4):   2%|▏         | 3225/159641 [00:02<01:02, 2492.89 examples/s]

... (211 outputs left out)


Map (num_proc=4):  73%|███████▎  | 13000/17738 [00:01<00:00, 13890.72 examples/s]


Map (num_proc=4):  98%|█████████▊| 17304/17738 [00:01<00:00, 18970.19 examples/s]


Map (num_proc=4): 100%|██████████| 17738/17738 [00:01<00:00, 10568.82 examples/s]

Blocks: 29815 train  | 2000 eval


In [10]:
# ╔════════ TPU v3-8 • 8 cores • LoRA training  (verbose) ══════════╗
import os, gc, time
os.environ["XRT_TPU_CONFIG"] = "localservice;0;localhost:51011"  # 8-core routing
os.environ["TRANSFORMERS_NO_TF"] = "1"                           # skip TensorFlow
gc.collect()

from accelerate import notebook_launcher
from transformers import (
    TrainingArguments, Trainer, default_data_collator,
    ProgressCallback
)
import torch_xla.core.xla_model as xm

# ── tiny helper so we see progress from *only* the master core ─────
class MasterPrinterCallback(ProgressCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if xm.is_master_ordinal():
            step = int(state.global_step)
            loss = logs.get("loss")
            lr   = logs.get("learning_rate")
            eta  = state.eta_remaining_str
            print(f"[step {step:>4}/{args.max_steps}] "
                  f"loss={loss:.4f}  lr={lr:.2e}  ETA={eta}")

def tpu_train():
    device = xm.xla_device()
    model.to(device)
    gc.collect(); xm.mark_step()

    args = TrainingArguments(
        output_dir                = "./llama3b-lora-tpu",
        max_steps                 = 1_000,          # ≈16 M tokens
        per_device_train_batch_size = 32,           # 32 × 8 = 256 global
        gradient_accumulation_steps = 1,
        learning_rate             = 2e-4,
        warmup_steps              = 100,
        bf16=True, fp16=False,
        optim="adamw_torch",
        logging_strategy="steps",
        logging_first_step=True,
        logging_steps=10,          # ← emits every 10 steps
        disable_tqdm=False,        # ← tqdm bar visible in cell output
        evaluation_strategy="no",
        save_strategy="no",
        report_to="none",
        remove_unused_columns=False,
        dataloader_num_workers=4,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=default_data_collator,
        callbacks=[MasterPrinterCallback],   # verbose logs
    )

    trainer.train()

    if xm.is_master_ordinal():
        out_dir = "./lora-tpu-adapter"
        trainer.save_model(out_dir)
        tokenizer.save_pretrained(out_dir)
        print(f"\n✅  Adapter saved to {out_dir}")

# launch 8 TPU workers; only core-0 prints
notebook_launcher(tpu_train, num_processes=8)


E0000 00:00:1750344891.415386      74 common_lib.cc:612] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: ===
learning/45eac/tfrc/runtime/common_lib.cc:230


In [ ]:
prompt = "I can't sleep and feel very anxious. Any quick coping ideas?"
inputs = tokenizer(prompt, return_tensors="pt")
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=60, temperature=0.7)
print(tokenizer.decode(out[0], skip_special_tokens=True))


In [ ]:
python peft/utils/merge_adapter.py \
  --base_model meta-llama/Llama-3.2-3B-Instruct \
  --adapter_path ./lora-tpu-adapter \
  --save_path    ./llama3b-merged-fp16

./quantize ./llama3b-merged-fp16/ggml-model-f16.gguf \
           ./llama3b-merged-q4.gguf q4_0
